# Live Dota 2 Win Probability
**Vexil Capital · Senior Data Scientist take-home**

Estimate the probability that Radiant wins at minute *t*, using only information at or before game clock 60*t seconds. Gold-only logistic regression is the compulsory baseline. Our small candidate family tests whether the *age, velocity, and objective conversion* of an economic lead change its predictive value.

**Research status:** provisional implementation, not a submitted result. The notebook is ready to run, but its final 500-match ID manifest and executed outputs must be frozen after the first verified pull. Every number below must come from execution; no results are prefilled.

Notebook sections: Setup → Data → Features → Split → Models → Evaluation. Only the OpenDota `/proMatches` and `/matches/{match_id}` endpoints are used.

Methodology: Quant Research Bible v1.4.1 §§1.2–1.4, 3.1–3.7, 4.3, 5.1–5.6, 6.5, 7.2–7.7, 12.1, 13.1, 14.1. The research is **prediction-only**, not a trading or causal study.


## 1 · Setup

For a local run, install pinned dependencies from `requirements.txt` before opening this notebook. A fresh Colab runtime installs inline-pinned analysis packages without requiring any other file. Kaggle uses its preinstalled scientific stack: changing NumPy/SciPy inside its already-running Python kernel can cause an ABI conflict. The notebook prints the actual runtime versions. Internet is required to download OpenDota matches. A cold data pull is rate-limited and may take more than ten minutes. No OpenDota or Kaggle API key is required to run the notebook.

Track author work time honestly before submission; elapsed kernel runtime is not a substitute for the hours actually spent researching and writing.


In [ ]:
import sys, subprocess, importlib.util, os
# Colab must be able to run without an external requirements.txt.
ON_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or os.path.isdir("/kaggle/working")
COLAB = not ON_KAGGLE and ("google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG")))
PINS = (
    "numpy==2.2.6", "pandas==2.3.3", "scipy==1.16.3",
    "scikit-learn==1.7.2", "matplotlib==3.10.7",
    "requests==2.32.5", "tqdm==4.67.1",
)
if COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "--quiet", "--disable-pip-version-check", *PINS])
import os, re, json, time, math, random, hashlib, gzip, tempfile, copy
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, brier_score_loss

SEED = 20261113
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_columns", 40)
AUTHOR_HOURS_SPENT = "To be recorded from actual work log before submission"
print("Python:", sys.version.split()[0], "| Colab:", COLAB, "| Kaggle:", ON_KAGGLE, "| seed:", SEED)
print("Actual hours invested:", AUTHOR_HOURS_SPENT)
print("Package versions:", {k: __import__(k).__version__
                            for k in ("numpy", "pandas", "sklearn", "matplotlib")})


## 2 · Data

Exactly 500 usable professional matches, spanning at least two patches. Eligibility follows the brief: at least ten minutes and a parsed `radiant_gold_adv` array. Selection never uses the winner.

**Frozen manifest gate:** `FROZEN_MATCH_IDS` is empty until the initial sample is verified. This exploratory mode is not submission-ready. Before freezing the submission, paste the 500 selected IDs **into this cell** and rerun from a fresh runtime. Every subsequent run fetches those same matches from OpenDota, rather than selecting a moving recent-match window. OpenDota can reparse historical records; upstream payload revisions remain a documented limitation.

Patch selection (revised from a **patch-metadata-only** preflight, before any model results): recent pro matches sampled over 30 pages were all patch 60, so pulling thousands of full match payloads just to locate the preceding patch would exhaust free API capacity. Instead, probe at most three eligible matches per proMatches page to locate the first patch transition; then select the **250 closest eligible games on each side** of that transition. Selection never uses match outcomes. This defines a deliberate two-patch transition cohort, **not** a random sample of all current pro games, and the report must state that generalization limit.


In [ ]:
BASE_URL = "https://api.opendota.com/api"
N_MATCHES = 500
COHORT_N = N_MATCHES // 2
MAX_PRO_PAGES = 110  # Discovery uses at most three parsed-match probes per page.
FROZEN_MATCH_IDS = ()  # Must contain exactly 500 match IDs before final delivery.
assert not FROZEN_MATCH_IDS or len(set(FROZEN_MATCH_IDS)) == N_MATCHES

class OpenDota:
    """Only the two permitted endpoints; disk cache is disposable runtime state."""
    def __init__(self, cache_dir=None, interval=1.26, max_retries=6):
        self.cache_dir = Path(cache_dir or Path(tempfile.gettempdir()) / "vexil_opendota_json_v1")
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.interval, self.max_retries = interval, max_retries
        self.next_call, self.calls, self.cache_hits = 0., 0, 0
        self.session = requests.Session()
        self.session.headers.update({"User-Agent": "vexil-takehome-research/1.0"})
    def get(self, path, params=None):
        assert path == "/proMatches" or re.fullmatch(r"/matches/\d+", path), path
        params = params or {}
        assert path != "/proMatches" or set(params).issubset({"less_than_match_id"})
        key = hashlib.sha256(json.dumps([path, params], sort_keys=True).encode()).hexdigest()
        file = self.cache_dir / (key + ".json.gz")
        if file.exists():
            self.cache_hits += 1
            with gzip.open(file, "rt", encoding="utf8") as handle:
                return json.load(handle)
        for attempt in range(self.max_retries):
            wait = self.next_call - time.monotonic()
            if wait > 0: time.sleep(wait)
            self.next_call = time.monotonic() + self.interval
            try:
                response = self.session.get(BASE_URL + path, params=params, timeout=(8, 35))
                self.calls += 1
                if response.status_code == 404:
                    raise ValueError("Match not found: " + path)
                if response.status_code in (429, 500, 502, 503, 504):
                    retry_after = response.headers.get("Retry-After", "")
                    try: delay = max(2., float(retry_after))
                    except ValueError: delay = min(60., 2. ** attempt)
                    time.sleep(min(delay + 0.1 * attempt, 90.))
                    continue
                response.raise_for_status()
                payload = response.json()
                with gzip.open(file, "wt", encoding="utf8") as handle:
                    json.dump(payload, handle, separators=(",", ":"))
                return payload
            except (requests.Timeout, requests.ConnectionError):
                time.sleep(min(2. ** attempt, 30.))
        raise RuntimeError("OpenDota failed after bounded retries: " + path)

def usable(match):
    return (isinstance(match, dict)
            and isinstance(match.get("radiant_gold_adv"), list)
            and len(match["radiant_gold_adv"]) > 0
            and isinstance(match.get("duration"), (int, float))
            and match["duration"] >= 600
            and isinstance(match.get("radiant_win"), bool))

def patch_coverage_ok(matches):
    """Exactly 250 usable matches on each side of one discovered patch boundary."""
    counts = Counter(str(m.get("patch")) for m in matches if m.get("patch") is not None)
    return len(matches) == N_MATCHES and sorted(counts.values()) == [COHORT_N, COHORT_N]

def collect_matches(api, frozen=()):
    if frozen:
        if len(frozen) != N_MATCHES or len(set(frozen)) != N_MATCHES:
            raise ValueError("Frozen manifest must contain 500 unique match IDs")
        data = [api.get("/matches/" + str(int(mid))) for mid in frozen]
        if not all(usable(m) for m in data):
            raise ValueError("Frozen manifest contains now-unusable matches; do not silently replace them")
        if not patch_coverage_ok(data):
            raise ValueError("Frozen manifest's 250/250 patch composition has changed")
        return data, []

    # The moving proMatches feed contains no patch field. A few label-blind
    # detail probes per page locate a patch transition WITHOUT downloading
    # thousands of match payloads from the same patch.
    pages, seen_ids, exclusions, excluded_ids = [], set(), [], set()
    cursor = None
    recent_patch, older_patch, boundary_page = None, None, None

    def next_page():
        nonlocal cursor
        if len(pages) >= MAX_PRO_PAGES:
            raise RuntimeError("Patch discovery exceeded bounded page budget")
        raw = api.get("/proMatches", {} if cursor is None else {"less_than_match_id": cursor})
        if not isinstance(raw, list) or not raw:
            raise RuntimeError("OpenDota exhausted proMatches before a valid two-patch cohort")
        page = []
        for row in raw:
            mid = int(row["match_id"])
            if mid not in seen_ids:
                page.append({"match_id": mid, "start_time": int(row.get("start_time") or 0)})
                seen_ids.add(mid)
        fresh_cursor = min(int(row["match_id"]) for row in raw)
        if cursor is not None and fresh_cursor >= cursor:
            raise RuntimeError("Nondecreasing proMatches pagination cursor")
        cursor = fresh_cursor
        pages.append(page)
        return page

    def note_exclusion(mid, reason):
        if mid not in excluded_ids:
            exclusions.append((mid, reason))
            excluded_ids.add(mid)

    def details(mid):
        try:
            return api.get("/matches/" + str(mid))
        except ValueError:
            note_exclusion(mid, "not_found")
            return {}

    while len(pages) < MAX_PRO_PAGES:
        page = next_page()
        sampled_patch = None
        # The first match on a new page may not yet have been parsed.
        for rec in page[:3]:
            match = details(rec["match_id"])
            if usable(match) and match.get("patch") is not None:
                sampled_patch = str(match["patch"])
                break
        if sampled_patch is None:
            print("patch discovery page", len(pages), "| no parsed patch probe")
            continue
        if recent_patch is None:
            recent_patch = sampled_patch
        elif sampled_patch != recent_patch:
            older_patch = sampled_patch
            boundary_page = len(pages) - 1
            break
        if len(pages) % 5 == 0:
            print("patch discovery:", len(pages), "pages, latest patch", recent_patch)
    if older_patch is None or boundary_page is None or not boundary_page:
        raise RuntimeError("No prior patch boundary found within the bounded discovery window")

    print("Observed adjacent patch cohorts:", older_patch, "->", recent_patch,
          "| transition near proMatches page", boundary_page + 1)

    # A preregistered, outcome-blind, balanced window around the discovered
    # boundary: 250 usable games closest to it on each side. Do not stratify
    # by Radiant wins, duration beyond eligibility, or model scores.
    new_group = []
    for rec in reversed([x for page in pages[:boundary_page] for x in page]):
        match = details(rec["match_id"])
        if usable(match) and str(match.get("patch")) == recent_patch:
            new_group.append(match)
            if len(new_group) == COHORT_N:
                break
        else:
            note_exclusion(rec["match_id"], "wrong_patch" if usable(match) else "unparsed_or_too_short")
    if len(new_group) != COHORT_N:
        raise RuntimeError("Insufficient eligible newer-patch matches near the boundary")

    old_group, page_idx = [], boundary_page
    while len(old_group) < COHORT_N:
        if page_idx >= len(pages):
            next_page()
        for rec in pages[page_idx]:
            match = details(rec["match_id"])
            if usable(match) and str(match.get("patch")) == older_patch:
                old_group.append(match)
                if len(old_group) == COHORT_N:
                    break
            else:
                note_exclusion(rec["match_id"], "wrong_patch" if usable(match) else "unparsed_or_too_short")
        page_idx += 1
    selected = old_group + new_group
    ids = [int(m["match_id"]) for m in selected]
    if len(ids) != N_MATCHES or len(set(ids)) != N_MATCHES or not patch_coverage_ok(selected):
        raise RuntimeError("Selected matches violate frozen 500-match patch-cohort contract")
    print("Selected", N_MATCHES, "eligible games with", COHORT_N, "per patch; API calls:", api.calls,
          "| excluded candidates:", len(exclusions))
    return selected, exclusions


In [ ]:
api = OpenDota()
matches, rejected = collect_matches(api, FROZEN_MATCH_IDS)
manifest = pd.DataFrame([{
    "match_id": int(m["match_id"]),
    "start_time": int(m["start_time"]),
    "patch": str(m.get("patch")),
    "series_id": int(m.get("series_id") or 0),
    "duration": int(m["duration"]),
    "radiant_win": int(m["radiant_win"]),
} for m in matches]).sort_values(["start_time", "match_id"]).reset_index(drop=True)
assert len(manifest) == N_MATCHES and manifest.match_id.is_unique
assert patch_coverage_ok(matches)
print("API calls:", api.calls, "| cache hits:", api.cache_hits, "| exclusions:", len(rejected))
print("Patch distribution:", manifest.patch.value_counts().to_dict())
print("Candidate exclusions by reason:", dict(Counter(reason for _, reason in rejected)))
display(manifest[["match_id", "start_time", "patch"]].assign(
    start_time=lambda x: pd.to_datetime(x.start_time, unit="s", utc=True)))
print("FROZEN_MATCH_IDS = " + repr(tuple(manifest.match_id.tolist())))


## 3 · Features and point-in-time falsification

The prediction cutoff at minute `t` is `60*t` seconds. Array access uses at most index `t`; events use `event.time <= 60*t`. Negative pre-horn event times are admitted, as specified in the brief. Source values from index `t` are assumed known at that minute **under the assignment's API convention**; OpenDota's retrospectively parsed response does not itself prove historical receive-time availability.

A final-match field, e.g. net worth, kill totals, duration or result, is **never** a predictor. Duration determines only how many minute rows exist. Missing values are not backfilled from future minutes. Objective scores are inferred only from building-destruction event keys with an identified building side; unrecognized objective events are ignored rather than guessed.


In [ ]:
FEATURE_COLUMNS = ("gold_k", "minute10", "gold_x_time", "xp_k", "gold_velocity5_k",
                   "xp_velocity5_k", "tower_delta", "barracks_delta", "roshan_delta",
                   "kill_delta")

def last_observed(values, t):
    if not isinstance(values, list): return float("nan")
    for i in range(min(t, len(values) - 1), -1, -1):
        v = values[i]
        if isinstance(v, (float, int)) and math.isfinite(v):
            return float(v)
    return float("nan")

def objective_snapshot(match, seconds):
    towers, barracks, roshan = 0, 0, 0
    for event in match.get("objectives") or []:
        if not isinstance(event, dict) or not isinstance(event.get("time"), (int, float)):
            continue
        if event["time"] > seconds: continue
        kind = event.get("type", "")
        if kind == "building_kill":
            key = str(event.get("key") or "").lower()
            # Destroyed Dire building -> Radiant gained an objective.
            delta = 1 if "badguys" in key else -1 if "goodguys" in key else 0
            if "tower" in key: towers += delta
            elif "rax" in key or "barracks" in key: barracks += delta
        elif kind == "CHAT_MESSAGE_ROSHAN_KILL":
            # Dota team IDs: 2=Radiant, 3=Dire; do not infer from victim/score totals.
            team = event.get("team")
            if team == 2: roshan += 1
            elif team == 3: roshan -= 1
    return towers, barracks, roshan

def kill_snapshot(match, seconds):
    delta = 0
    for player in match.get("players") or []:
        slot = player.get("player_slot")
        if not isinstance(slot, int): continue
        sign = 1 if slot < 128 else -1
        for event in player.get("kills_log") or []:
            if isinstance(event, dict) and isinstance(event.get("time"), (int, float)):
                if event["time"] <= seconds: delta += sign
    return delta

def snapshot_features(match, t):
    assert isinstance(t, int) and t >= 0
    gold = last_observed(match.get("radiant_gold_adv"), t)
    if not math.isfinite(gold):
        raise ValueError("No usable gold lead as of minute %d" % t)
    xp = last_observed(match.get("radiant_xp_adv"), t)
    old_gold = last_observed(match.get("radiant_gold_adv"), max(0, t - 5))
    old_xp = last_observed(match.get("radiant_xp_adv"), max(0, t - 5))
    towers, barracks, roshan = objective_snapshot(match, 60 * t)
    x = {
        "gold_k": gold / 1000.,
        "minute10": t / 10.,
        "gold_x_time": (gold / 1000.) * np.log1p(t / 10.),
        "xp_k": xp / 1000.,
        "gold_velocity5_k": (gold - old_gold) / 1000.,
        "xp_velocity5_k": (xp - old_xp) / 1000. if math.isfinite(xp) and math.isfinite(old_xp) else float("nan"),
        "tower_delta": towers,
        "barracks_delta": barracks,
        "roshan_delta": roshan,
        "kill_delta": kill_snapshot(match, 60 * t),
    }
    assert set(x) == set(FEATURE_COLUMNS)
    return x

def make_minute_table(games):
    frames = []
    for match in games:
        max_minute = max(0, int(match["duration"]) // 60)
        for t in range(max_minute + 1):
            x = snapshot_features(match, t)
            frames.append({"match_id": int(match["match_id"]), "minute": t,
                           "radiant_win": int(match["radiant_win"]), **x})
    return pd.DataFrame.from_records(frames)

minute_table = make_minute_table(matches)
assert minute_table[["match_id", "minute"]].duplicated().sum() == 0
assert len(minute_table.match_id.unique()) == N_MATCHES
assert (minute_table.groupby("match_id").radiant_win.nunique() == 1).all()
print("Minute rows:", len(minute_table), "| unique matches:", minute_table.match_id.nunique())
display(minute_table.head(10))


In [ ]:
def prefix_truncated(match, t):
    shortened = copy.deepcopy(match)
    for k in ("radiant_gold_adv", "radiant_xp_adv"):
        if isinstance(shortened.get(k), list): shortened[k] = shortened[k][:t+1]
    shortened["objectives"] = [e for e in (shortened.get("objectives") or [])
                               if isinstance(e, dict) and isinstance(e.get("time"), (int, float)) and e["time"] <= 60*t]
    for player in shortened.get("players") or []:
        for k in ("gold_t", "xp_t", "lh_t"):
            if isinstance(player.get(k), list): player[k] = player[k][:t+1]
        player["kills_log"] = [e for e in (player.get("kills_log") or [])
                               if isinstance(e, dict) and isinstance(e.get("time"), (int, float)) and e["time"] <= 60*t]
        for k in ("net_worth", "kills", "tower_kills", "total_gold", "total_xp"):
            player[k] = -999999
    shortened["radiant_win"] = not bool(shortened.get("radiant_win"))
    shortened["duration"] = -999999
    return shortened

def assert_same_features(a, b):
    assert a.keys() == b.keys()
    for key in a:
        x, y = a[key], b[key]
        assert (pd.isna(x) and pd.isna(y)) or np.isclose(x, y, rtol=0, atol=1e-10), (key, x, y)

# A deterministic stratified-in-time spot check over multiple matches and minutes.
checked = 0
# The selected data are ordered older-patch then newer-patch. Test both.
pit_cases = matches if len(matches) <= 12 else matches[:6] + matches[-6:]
for match in pit_cases:
    for t in sorted(set([0, 1, 9, 10, 20, min(int(match["duration"])//60, 30)])):
        assert_same_features(snapshot_features(match, t),
                             snapshot_features(prefix_truncated(match, t), t))
        checked += 1
print("PIT prefix/label poisoning PASS:", checked, "| patches:", sorted({str(m.get("patch")) for m in pit_cases}))


## 4 · Split

The deployment analogy is forward evaluation on later professional games. Sort by start time, then choose two chronological cutoffs that **never divide a series**. Each minute of a match inherits its match-level assignment. No random minute-row split or global preprocessing fit is permitted. A chronological holdout can be harder across patch shifts; that shift is evidence to report, not something to hide by reshuffling.


In [ ]:
def series_safe_split(info, train_target=0.60, val_target=0.20):
    info = info.sort_values(["start_time", "match_id"]).reset_index(drop=True).copy()
    n = len(info)
    groups = [("series", int(sid)) if int(sid) > 0 else ("match", int(mid))
              for sid, mid in zip(info.series_id, info.match_id)]
    mins, maxs = {}, {}
    for i, g in enumerate(groups):
        mins[g] = min(i, mins.get(g, i))
        maxs[g] = max(i, maxs.get(g, i))
    cuts = [i for i in range(1, n)
            if all(not (mins[g] < i <= maxs[g]) for g in mins)]
    min_block = max(25, int(n * 0.10))
    pairs = [(i, j) for i in cuts for j in cuts
             if min_block <= i and min_block <= (j-i) and min_block <= (n-j)]
    if not pairs: raise RuntimeError("No valid chronological series-safe boundaries")
    a, b = min(pairs, key=lambda p: (abs(p[0]-n*train_target) + abs(p[1]-n*(train_target+val_target)),
                                      abs(p[1]-n*(train_target+val_target)), p))
    info["partition"] = np.select([info.index < a, info.index < b],
                                  ["train", "validation"], default="test")
    ids = {split: set(info.loc[info.partition == split, "match_id"]) for split in ("train", "validation", "test")}
    assert not (ids["train"] & ids["validation"] or ids["train"] & ids["test"] or ids["validation"] & ids["test"])
    grouped = info.groupby(["series_id", "partition"]).size().reset_index()
    for sid, sub in grouped.groupby("series_id"):
        if sid > 0: assert len(sub) == 1, ("series leak", sid)
    assert info.loc[info.partition == "train", "start_time"].max() <= info.loc[info.partition == "validation", "start_time"].min()
    assert info.loc[info.partition == "validation", "start_time"].max() <= info.loc[info.partition == "test", "start_time"].min()
    return info

manifest = series_safe_split(manifest)
data = minute_table.merge(manifest[["match_id", "partition", "patch"]], on="match_id",
                          validate="many_to_one")
split_counts = data.groupby("partition").agg(
    matches=("match_id", "nunique"), minute_rows=("minute", "size"),
    radiant_win_rate=("radiant_win", "mean"))
display(split_counts)
display(pd.crosstab(manifest.partition, manifest.patch))
train = data[data.partition == "train"].copy()
validation = data[data.partition == "validation"].copy()
test = data[data.partition == "test"].copy()


## 5 · Models and finite search

**M0** is a gold-lead-only logistic-regression baseline, with an intercept. This is exactly the required baseline: no time, XP or other inputs.

The proposed candidate family is limited to three predeclared nested specifications (no hero statistics, no external codes or other Dota endpoints):

- **M1 (time-value):** M0 + elapsed time and gold × log elapsed time. Does the same lead mean something different at different minutes?
- **M2 (lead fragility):** M1 + XP and trailing five-minute changes in gold and XP. Does momentum add information beyond the level?
- **M3 (conversion):** M2 + timestamp-filtered towers, barracks, Roshan and kill events. Does realized map control contribute?

The top-two-player gold concentration feature is computed for descriptive investigation only; it does not enter the finite model search. All preprocessing is fitted **only on training matches**. Choose the proposed model by validation log loss, with Brier score reported as a second check. Calibration is diagnosed, not retrofitted on the test set.


In [ ]:
SETS = {
    "M0_gold_only": ["gold_k"],
    "M1_time_value": ["gold_k", "minute10", "gold_x_time"],
    "M2_lead_fragility": ["gold_k", "minute10", "gold_x_time", "xp_k",
                          "gold_velocity5_k", "xp_velocity5_k"],
    "M3_objective_conversion": ["gold_k", "minute10", "gold_x_time", "xp_k",
                                "gold_velocity5_k", "xp_velocity5_k",
                                "tower_delta", "barracks_delta", "roshan_delta", "kill_delta"],
}
def fit_model(frame, columns):
    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("scale", StandardScaler()),
        ("logistic", LogisticRegression(C=1., max_iter=1000, random_state=SEED)),
    ])
    model.fit(frame[columns], frame["radiant_win"])
    return model

models = {name: fit_model(train, cols) for name, cols in SETS.items()}
val_results = []
for name, model in models.items():
    p = model.predict_proba(validation[SETS[name]])[:, 1]
    val_results.append({"model": name,
                        "log_loss": log_loss(validation.radiant_win, p, labels=[0, 1]),
                        "brier": brier_score_loss(validation.radiant_win, p)})
val_table = pd.DataFrame(val_results).sort_values("log_loss")
display(val_table)
selected = val_table[val_table.model != "M0_gold_only"].iloc[0]["model"]
print("PROPOSED MODEL CHOSEN USING VALIDATION ONLY:", selected)
print("The final test set has not entered model fitting or selection.")


## 6 · Evaluation on the untouched held-out matches

The primary estimand is average proper loss over **minute-level forecast opportunities**. Longer games contribute more rows. As a sensitivity analysis, also report each match's mean loss weighted equally across matches. The confidence interval procedure always resamples **whole matches** and uses identical resamples for the model and baseline.

Game phases: 0–9, 10–19, 20–29 and ≥30 minutes. Intervals are percentile bootstrap 95% intervals, not guaranteed population bounds under patch drift.


In [ ]:
base_prob = models["M0_gold_only"].predict_proba(test[SETS["M0_gold_only"]])[:, 1]
model_prob = models[selected].predict_proba(test[SETS[selected]])[:, 1]
predictions = test[["match_id", "minute", "radiant_win", "patch"]].copy().reset_index(drop=True)
predictions["p_model"] = model_prob
predictions["p_baseline"] = base_prob
assert predictions[["p_model", "p_baseline"]].notna().all().all()
assert predictions[["p_model", "p_baseline"]].ge(0.).all().all()
assert predictions[["p_model", "p_baseline"]].le(1.).all().all()
predictions["phase"] = pd.cut(predictions.minute,
                              bins=[-1, 9, 19, 29, np.inf],
                              labels=["0–9", "10–19", "20–29", "30+"])
print("Held-out matches:", predictions.match_id.nunique(),
      "| predictions:", len(predictions),
      "| radiant win rate (match-level):",
      manifest.loc[manifest.partition == "test", "radiant_win"].mean())

EPS = 1e-15
y = predictions.radiant_win.to_numpy()
for kind in ("model", "baseline"):
    p = np.clip(predictions["p_" + kind].to_numpy(), EPS, 1-EPS)
    predictions["logloss_" + kind] = -(y * np.log(p) + (1-y) * np.log1p(-p))
    predictions["brier_" + kind] = (p-y) ** 2

def match_bootstrap(frame, loss_type, n_boot=2000, seed=SEED):
    # Pairing is enforced by resampling the same match indices for both columns.
    aggregates = frame.groupby("match_id", sort=True).agg(
        n=("minute", "size"),
        model=("{}_model".format(loss_type), "sum"),
        baseline=("{}_baseline".format(loss_type), "sum"),
    )
    if aggregates.empty: raise ValueError("Empty phase: cannot estimate confidence interval")
    vec = aggregates[["n", "model", "baseline"]].to_numpy(dtype=float)
    n = len(vec)
    rng = np.random.default_rng(seed)
    indices = rng.integers(0, n, size=(n_boot, n))
    samples = vec[indices].sum(axis=1)
    model_vals = samples[:, 1] / samples[:, 0]
    baseline_vals = samples[:, 2] / samples[:, 0]
    actual = vec.sum(axis=0)
    metrics = {"model": actual[1]/actual[0],
               "baseline": actual[2]/actual[0],
               "difference": (actual[1]-actual[2])/actual[0]}
    for label, arr in (("model", model_vals), ("baseline", baseline_vals),
                       ("difference", model_vals-baseline_vals)):
        metrics[label + "_lo"], metrics[label + "_hi"] = np.quantile(arr, [0.025, 0.975])
    metrics["matches"] = n
    metrics["rows"] = len(frame)
    return metrics

results = []
for phase in ["overall", "0–9", "10–19", "20–29", "30+"]:
    window = predictions if phase == "overall" else predictions[predictions.phase == phase]
    for metric in ("logloss", "brier"):
        results.append({"phase": phase, "metric": metric,
                        **match_bootstrap(window, metric)})
evaluation = pd.DataFrame(results)
display(evaluation.round(5))
print("Difference = proposed model minus baseline; negative values favor the proposed model.")


In [ ]:
def reliability_plot(frame, probability, ax, bins=np.linspace(0, 1, 11)):
    p = frame[probability].to_numpy()
    y = frame["radiant_win"].to_numpy()
    idx = np.clip(np.digitize(p, bins, right=False)-1, 0, len(bins)-2)
    points = []
    for j in range(len(bins)-1):
        mask = idx == j
        if not mask.any(): continue
        points.append((p[mask].mean(), y[mask].mean(), mask.sum()))
    pts = np.asarray(points)
    ax.plot([0,1],[0,1], "--", color="0.5", linewidth=1)
    if len(points):
        ax.plot(pts[:,0],pts[:,1],marker="o",label=probability)
        for observed, empirical, count in pts:
            ax.annotate(str(int(count)),(observed,empirical),
                        xytext=(0,5),textcoords="offset points",fontsize=8,ha="center")
    ax.set(xlim=(0,1),ylim=(0,1),xlabel="Predicted Radiant win probability",
           ylabel="Observed Radiant win frequency",title=probability)
    ax.grid(alpha=0.2)
    return pts

fig, axs = plt.subplots(1, 2, figsize=(11, 4.5))
model_bins = reliability_plot(predictions, "p_model", axs[0])
baseline_bins = reliability_plot(predictions, "p_baseline", axs[1])
fig.suptitle("Held-out reliability: labels denote minute-row counts per fixed bin")
fig.tight_layout()
plt.show()
print("Note: minutes within a match share one final outcome. Bin counts are not independent outcomes.")


In [ ]:
def equal_match_weight_loss(frame, kind):
    by_match = frame.groupby("match_id")[["logloss_" + kind, "brier_" + kind]].mean()
    return {"log_loss": by_match["logloss_" + kind].mean(),
            "brier": by_match["brier_" + kind].mean()}
print("Equal-match-weighted sensitivity:")
display(pd.DataFrame({
    "proposed": equal_match_weight_loss(predictions, "model"),
    "baseline": equal_match_weight_loss(predictions, "baseline"),
}).round(5))
print("By held-out patch (descriptive; small subgroups can be noisy):")
display(predictions.groupby("patch").agg(
    matches=("match_id", "nunique"),
    rows=("minute", "size"),
    proposed_logloss=("logloss_model", "mean"),
    baseline_logloss=("logloss_baseline", "mean"),
    proposed_brier=("brier_model", "mean"),
    baseline_brier=("brier_baseline", "mean"),
).round(5))


### Inspecting the time-value and comeback hypotheses

These are **descriptive observations**, not causal effects and not trading signals. Inspect the fitted time-interaction coefficients, then show the model's conditional forecast at identical gold leads across game minutes. Scenario values are artificial and can be outside the data manifold; do not interpret them as interventions.

A separate trajectory visualizes one held-out match with the largest raw gold-lead reversal **chosen without using model error**. It illustrates why point-in-time forecasts can change even when the final label is fixed.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
scenario_minutes = np.arange(0, 51, 2)
for gold in (-6, -3, 0, 3, 6):
    scenarios = pd.DataFrame([{"gold_k": gold, "minute10": t/10,
                                "gold_x_time": gold*np.log1p(t/10)} for t in scenario_minutes])
    if selected == "M1_time_value":
        p = models[selected].predict_proba(scenarios[SETS[selected]])[:, 1]
    else:
        # Remaining features fixed at the training median; scenarios are illustrative.
        full = pd.DataFrame({col: np.full(len(scenarios), train[col].median())
                             for col in SETS[selected]})
        for col in scenarios: full[col] = scenarios[col]
        p = models[selected].predict_proba(full[SETS[selected]])[:, 1]
    ax.plot(scenario_minutes, p, label=f"Gold lead {gold:+d}k")
ax.set(xlabel="Match minute", ylabel="Model P(Radiant wins)",
       title="Time-value scenarios (conditional associations, not interventions)",ylim=(0,1))
ax.legend(ncol=3, fontsize=8)
ax.grid(alpha=0.2)
plt.show()

# Case study chosen by the gold process only, not by realized outcome/prediction accuracy.
g = test[["match_id", "minute", "gold_k"]].copy()
ranked = g.groupby("match_id")["gold_k"].agg(lambda a: a.max()-a.min()).sort_values(ascending=False)
ex_id = int(ranked.index[0])
chosen = test[test.match_id == ex_id].sort_values("minute")
curve = predictions[predictions.match_id == ex_id].sort_values("minute")
fig, axs = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axs[0].plot(chosen.minute, chosen.gold_k)
axs[0].axhline(0, color="0.55", linewidth=1)
axs[0].set(ylabel="Radiant gold lead (thousands)",title=f"Illustrative held-out game {ex_id}")
axs[1].plot(curve.minute, curve.p_model, label="Proposed model")
axs[1].plot(curve.minute, curve.p_baseline, label="Gold-only baseline",alpha=0.8)
axs[1].set(xlabel="Match minute",ylabel="P(Radiant wins)",ylim=(0,1))
axs[1].legend()
for a in axs: a.grid(alpha=0.2)
fig.tight_layout()
plt.show()


### A fixed-minute comeback diagnostic

Predefined before seeing held-out outcomes: at minute 20, among games where one side leads by at least 3,000 gold, compare that leader's eventual loss rate between **deteriorating** and **non-deteriorating** five-minute gold momentum. Each match contributes at most one observation. The cutoff is descriptive and is **not** used to select or fit a model.

Small subgroups are expected; report counts and Wilson intervals (independent-game approximation), not a causal claim. Within-series dependence, patch mix, draft strength and win-state confounding remain unaddressed. This is a limited observation, not evidence that changing momentum causes a comeback (Bible v1.4.1 §§1.2–1.4, 13.1–13.2).


In [ ]:
def wilson_95(k, n):
    if n <= 0:
        return float("nan"), float("nan")
    p, z = k / n, 1.959963984540054
    den = 1 + z*z/n
    center = (p + z*z/(2*n)) / den
    radius = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / den
    return center-radius, center+radius

minute20 = test.loc[(test.minute == 20) & (test.gold_k.abs() >= 3.0),
                    ["match_id", "gold_k", "gold_velocity5_k", "radiant_win", "patch"]].copy()
minute20["lead_deteriorating"] = minute20.gold_k * minute20.gold_velocity5_k < 0
minute20["leader_lost"] = np.where(minute20.gold_k > 0,
                                   1-minute20.radiant_win, minute20.radiant_win).astype(int)
summary = []
for flag, group in minute20.groupby("lead_deteriorating", dropna=False):
    n = int(len(group))
    k = int(group.leader_lost.sum())
    lo, hi = wilson_95(k, n)
    summary.append({"lead_deteriorating": bool(flag), "matches": n,
                    "eventual_leader_losses": k, "loss_rate": k/n,
                    "rate_95_lo": lo, "rate_95_hi": hi})
comeback_table = pd.DataFrame(summary)
print("Fixed minute-20 comeback diagnostic (threshold |gold lead| >= 3,000):")
print("Eligible held-out matches:", len(minute20),
      "| Rate intervals assume independent games; series dependence may widen them.")
display(comeback_table.round(4) if not comeback_table.empty else
        pd.DataFrame({"note": ["No held-out minute-20 snapshots met the preregistered threshold."]}))
if not comeback_table.empty:
    fig, ax = plt.subplots(figsize=(7, 3.5))
    xpos = np.arange(len(comeback_table))
    vals = comeback_table.loss_rate.to_numpy()
    errors = np.vstack((vals-comeback_table.rate_95_lo.to_numpy(),
                        comeback_table.rate_95_hi.to_numpy()-vals))
    ax.bar(xpos, vals, color="0.5", width=0.55)
    ax.errorbar(xpos, vals, yerr=errors, fmt="none", ecolor="black", capsize=4)
    labels = ["Lead weakening" if v else "Stable / increasing"
              for v in comeback_table.lead_deteriorating]
    ax.set_xticks(xpos, labels)
    ax.set(ylabel="Fraction of gold leaders that eventually lost",
           title="Minute 20: descriptive comeback risk", ylim=(0, 1))
    for x, n in zip(xpos, comeback_table.matches):
        ax.text(x, min(0.96, vals[x]+errors[1,x]+0.035), f"n={n}", ha="center", fontsize=8)
    fig.tight_layout()
    plt.show()


### Full held-out prediction table

Every test observation is retained below with `match_id`, `minute`, `p_model`, `p_baseline`, and `radiant_win`. The complete table is printed for saved-output inspection, not merely its head. A CSV can also be exported by the notebook user from the in-memory `predictions` DataFrame. No local data files are required as inputs.


In [ ]:
pd.set_option("display.max_rows", None)
final_predictions = predictions[["match_id", "minute", "p_model", "p_baseline", "radiant_win"]]
print("HELD-OUT PREDICTIONS:", len(final_predictions), "rows")
display(final_predictions)
pd.reset_option("display.max_rows")


## Conclusions and report hand-off

Before submission, interpret **actual** held-out confidence intervals, phase reliability, patch composition, and unsuccessful ablations in the separate ≤4-page report. The model is not automatically superior because it uses more features. If the test improvement is smaller than the resampled uncertainty or calibration is unstable, prefer the simple baseline for deployment.

**Limits:** 500 matches, shared outcomes across repeated minute forecasts, unobserved live API ingestion latency, possible OpenDota reparsing, patch/series dependence and no demonstration of profitability. AI assistance and any public code used must be disclosed accurately in the report. No external predictor code is used in this notebook.
